# Solicitação Dimap 1

#### **Solicitado por**: Bruno Carano

Será que você consegue me ajudar gerando mais 4 colunas adicionais nesse mesmo shape?

Seriam elas:

1 - CODLSQ (campo tipo texto; 6 caracteres) = junção de código do logradouro + dígito do código do logradouro + setor + quadra  ... todos da tabela dado avaliativo

2 - tp_uso_imovel : código de uso da tabela dado avaliativo

3 - tp_padrao_construcao : código de padrão construtivo da tabela dado avaliativo

4 - tp_zonauso: nesse colocar o código do tipo de zona de uso que predomina no lote, baseado no seu estudo do zoneamento que fez para a unidade




Basicamente preciso abrir o shapefile de lotes na intranet e acrescentar dados conforme solictado pelo sr. Bruno Carano. Os dados originais que precisam ser enriquecidos já foram subidos pelo sr. Bruno em uma pasta da intranet

In [1]:
import os
import pandas as pd
import geopandas as gpd
from utils.io.parquet import load_parquet

Abrindo arquivos

In [2]:
caminho_rede = r'\\sf_fs_cluster.sf.prefeitura.sp.gov.br\SF\SUREM\DECAD\DIMAP\DIMAP-1\Henrique'

lotes = os.path.join(caminho_rede, 'Lotes Fiscais')
dados_avaliativos = os.path.join(caminho_rede, 'dados_avaliativos.csv')

In [3]:
os.listdir(caminho_rede)

['dados_avaliativos.csv', 'Lotes Fiscais']

In [4]:
fpath = os.path.join(lotes, 'Lotes Fiscais.shp')
assert os.path.exists(fpath) and os.path.isfile(fpath), f"Arquivo não encontrado: {fpath}"

In [6]:
gdf = gpd.read_file(fpath)

In [7]:
gdf.head()

,id,cod_id,cod_setor,tipo_qp,cod_qp,tipo_lote,cod_cond,cod_lote,cod_status,geometry
0,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73..."
1,2.0,8897615,026,F,029,F,02,0000,1,"POLYGON ((336599.851 7396011.968, 336599.515 7..."
2,3.0,8897616,026,F,029,F,00,0004,1,"POLYGON ((336607.867 7396035.303, 336608.524 7..."
3,4.0,8897617,026,F,029,F,00,0005,1,"POLYGON ((336615.334 7396026.727, 336613.723 7..."
4,5.0,8897618,026,F,029,F,00,0030,1,"POLYGON ((336621.614 7395998.454, 336617.064 7..."


In [8]:
gdf.shape

(1692753, 10)

In [21]:
colunas = [
   'cd_contribuinte',
 'cd_logradouro',
 'tp_terreno',
 'qt_esquina_imovel',
 'qt_area_terreno_imovel',
 'qt_testada_terreno_calculo',
 'cd_condominio_imovel',
 'nr_fracao_ideal_terreno',
 'qt_area_construida_total',
 'tp_padrao_construcao',
 'qt_area_ocupada',
 'qt_pavimento',
 'an_construcao_corrigido',
 'tp_uso_imovel',
 'tp_meio_cobranca_tributo',
 'cd_zona_fiscal',
 'qt_primeira_testada_imovel'
]
df = pd.read_csv(dados_avaliativos, sep=';', names=colunas, header=None)

In [22]:
df.head()

,cd_contribuinte,cd_logradouro,tp_terreno,qt_esquina_imovel,qt_area_terreno_imovel,qt_testada_terreno_calculo,cd_condominio_imovel,nr_fracao_ideal_terreno,qt_area_construida_total,tp_padrao_construcao,qt_area_ocupada,qt_pavimento,an_construcao_corrigido,tp_uso_imovel,tp_meio_cobranca_tributo,cd_zona_fiscal,qt_primeira_testada_imovel
0,100300014,38121,2,1,136,13.00,0,0.0,135,32,108,1,1924,40,11,1,0
1,100300022,38121,1,0,90,6.00,0,0.0,67,32,67,1,1944,40,11,1,0
2,100300030,38121,1,0,105,7.85,0,0.0,140,32,84,2,1965,40,11,1,0
3,100300049,38121,1,0,108,6.05,0,0.0,103,32,86,1,1944,40,11,1,0
4,100300057,38121,1,0,120,6.70,0,0.0,170,32,110,2,2004,40,11,1,0


Passo 1- criando coluna código no gdf:


1 - CODLSQ (campo tipo texto; 6 caracteres) = junção de código do logradouro + dígito do código do logradouro + setor + quadra  ... todos da tabela dado avaliativo

Nesse caso é mais simples porque nao requer join é só usar o próprio df do dado avaliativo e depois dar join.

In [30]:
df.dtypes

cd_contribuinte                   str
cd_logradouro                   int64
tp_terreno                      int64
qt_esquina_imovel               int64
qt_area_terreno_imovel          int64
qt_testada_terreno_calculo    float64
cd_condominio_imovel            int64
nr_fracao_ideal_terreno       float64
qt_area_construida_total        int64
tp_padrao_construcao            int64
qt_area_ocupada                 int64
qt_pavimento                    int64
an_construcao_corrigido         int64
tp_uso_imovel                   int64
tp_meio_cobranca_tributo        int64
cd_zona_fiscal                  int64
qt_primeira_testada_imovel      int64
dtype: object

In [29]:
# precisa dar zfill porque abriu como int o cd contribuinte
df['cd_contribuinte'] = df['cd_contribuinte'].astype(str).str.zfill(11)

In [33]:
df['cd_logradouro'] = df['cd_logradouro'].astype(str).str.zfill(6)

In [40]:
df['setor'] = df['cd_contribuinte'].str[:3]
df['quadra'] = df['cd_contribuinte'].str[3:6]

In [42]:
df['CODLSQ']= df['cd_logradouro'] + '.' + df['setor'] + '.' + df['quadra']

In [46]:
join_cod_logradouro_quadra = df[['CODLSQ', 'cd_logradouro', 'setor', 'quadra']].drop_duplicates()
join_cod_logradouro_quadra.head()

,CODLSQ,cd_logradouro,setor,quadra
0,038121.001.003,038121,001,003
13,061565.001.003,061565,001,003
23,104850.001.003,104850,001,003
31,189936.001.003,189936,001,003
124,038121.001.004,038121,001,004


In [47]:
join_cod_logradouro_quadra['join_id'] = join_cod_logradouro_quadra['setor'] + '.' + join_cod_logradouro_quadra['quadra']

In [53]:
join_cod_logradouro_quadra.drop(['cd_logradouro', 'setor', 'quadra'], axis=1, inplace=True)

In [51]:
gdf['join_id'] = gdf['cod_setor'].str.zfill(3) + '.' + gdf['cod_qp'].str.zfill(3)

In [55]:
gdf = pd.merge(gdf, join_cod_logradouro_quadra, on='join_id', how='left')

In [56]:
gdf.head()

,id,cod_id,cod_setor,tipo_qp,cod_qp,tipo_lote,cod_cond,cod_lote,cod_status,geometry,join_id,CODLSQ
0,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",026.029,047368.026.029
1,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",026.029,106216.026.029
2,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",026.029,096776.026.029
3,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",026.029,043745.026.029
4,2.0,8897615,026,F,029,F,02,0000,1,"POLYGON ((336599.851 7396011.968, 336599.515 7...",026.029,047368.026.029


In [57]:
gdf['CODLSQ'].isnull().any()

np.True_

In [58]:
gdf['CODLSQ'].isnull().mean()

np.float64(0.00027265047299963337)

In [59]:
gdf.drop(columns=['join_id'], inplace=True)

2 - tp_uso_imovel : código de uso da tabela dado avaliativo


In [60]:
df.head()

,cd_contribuinte,cd_logradouro,tp_terreno,qt_esquina_imovel,qt_area_terreno_imovel,qt_testada_terreno_calculo,cd_condominio_imovel,nr_fracao_ideal_terreno,qt_area_construida_total,tp_padrao_construcao,qt_area_ocupada,qt_pavimento,an_construcao_corrigido,tp_uso_imovel,tp_meio_cobranca_tributo,cd_zona_fiscal,qt_primeira_testada_imovel,setor,quadra,CODLSQ
0,00100300014,038121,2,1,136,13.00,0,0.0,135,32,108,1,1924,40,11,1,0,001,003,038121.001.003
1,00100300022,038121,1,0,90,6.00,0,0.0,67,32,67,1,1944,40,11,1,0,001,003,038121.001.003
2,00100300030,038121,1,0,105,7.85,0,0.0,140,32,84,2,1965,40,11,1,0,001,003,038121.001.003
3,00100300049,038121,1,0,108,6.05,0,0.0,103,32,86,1,1944,40,11,1,0,001,003,038121.001.003
4,00100300057,038121,1,0,120,6.70,0,0.0,170,32,110,2,2004,40,11,1,0,001,003,038121.001.003


In [61]:
coluna_interesse = 'tp_uso_imovel'

Nesse caso vai precisar separar condominio de nao condominio para fazer o join

In [63]:
df['cd_condominio_imovel'].isnull().any()

np.False_

In [71]:
df['cd_condominio_imovel'].astype(str).str.len().unique()

array([1, 2, 3])

In [78]:
df[df['cd_condominio_imovel']>100]['cd_condominio_imovel'].astype(str).str[0].value_counts()

cd_condominio_imovel
1    174471
2     23061
3      4753
4       477
Name: count, dtype: int64

In [80]:
df['cd_condominio_imovel'].max()

np.int64(434)

In [64]:
df['is_condominio'] = df['cd_condominio_imovel']!=0

In [69]:
def id_lote_cond(row):
    if not row['is_condominio']:
        return row['cd_contribuinte']+'.'+row['cd_condominio_imovel'].astype(str).str.zfill(2)
    else:
        contrib = row['contribuinte']
        setor = contrib[:3]
        quadra = contrib[3:6]
        digito_sql = contrib[-1]
        contribuinte_formatado = setor+quadra+'0000'+digito_sql
        return contribuinte_formatado+'.'+row['cd_condominio_imovel'].astype(str).str.zfill(2)


In [62]:
gdf.head()

,id,cod_id,cod_setor,tipo_qp,cod_qp,tipo_lote,cod_cond,cod_lote,cod_status,geometry,CODLSQ
0,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",047368.026.029
1,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",106216.026.029
2,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",096776.026.029
3,1.0,8897614,026,F,029,F,01,0000,1,"POLYGON ((336597.97 7396029.548, 336598.446 73...",043745.026.029
4,2.0,8897615,026,F,029,F,02,0000,1,"POLYGON ((336599.851 7396011.968, 336599.515 7...",047368.026.029


In [74]:
gdf['cod_cond'].max()

'41'

In [72]:
gdf['cod_cond'].str.len().unique()

array([2])

In [68]:
gdf['tipo_lote'].unique()

<ArrowStringArray>
['F', 'V', 'M', 'S', 'D', 'W', 'E']
Length: 7, dtype: str